# Ripple Field Simulation (3D, Multiple Nodes)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

grid_size = 100
timesteps = 100
D = 0.1
lambda_decay = 10
ripple_strength = 5.0
coherence_threshold = 0.8

field = np.zeros((grid_size, grid_size))
x = np.arange(grid_size)
y = np.arange(grid_size)
X, Y = np.meshgrid(x, y)

nodes = [
    {'position': (30, 30), 'coherence': 0.9, 'entropy_gradient': np.array([1.0, 0.0])},
    {'position': (70, 30), 'coherence': 0.85, 'entropy_gradient': np.array([0.0, 1.0])},
    {'position': (50, 70), 'coherence': 0.95, 'entropy_gradient': np.array([-0.7, -0.7])},
]

def emit_multiple_ripples():
    total_ripple = np.zeros((grid_size, grid_size))
    for node in nodes:
        if node['coherence'] >= coherence_threshold:
            dx = X - node['position'][0]
            dy = Y - node['position'][1]
            distance = np.sqrt(dx**2 + dy**2)
            decay = np.exp(-distance / lambda_decay)
            direction = node['entropy_gradient'][0] * dx + node['entropy_gradient'][1] * dy
            ripple = ripple_strength * decay * direction / (distance + 1e-5)
            total_ripple += ripple
    return total_ripple

def update_field(field):
    laplacian = (
        np.roll(field, 1, axis=0) +
        np.roll(field, -1, axis=0) +
        np.roll(field, 1, axis=1) +
        np.roll(field, -1, axis=1) -
        4 * field
    )
    field += D * laplacian
    return field

for _ in range(timesteps):
    field = update_field(field)
    field += emit_multiple_ripples()

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')
ax.plot_surface(X, Y, field, cmap='plasma', edgecolor='none')
ax.set_title('3D Ripple Field from Multiple Nodes')
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Field Intensity')
plt.tight_layout()
plt.show()